In [1]:
# ============================================================
# Count Item Metadata rows linked to User Reviews
# Date range: 2020-01-01 to 2022-12-31
# Amazon Reviews 2023
#
# Logic:
# 1. Find parent_asin values from User Reviews in the date range.
# 2. Count Item Metadata rows whose parent_asin is in that set.
# ============================================================

!pip -q install pandas requests tqdm

import json
import requests
import pandas as pd
from tqdm.auto import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# ----------------------------
# Config
# ----------------------------

DATASET = "McAuley-Lab/Amazon-Reviews-2023"

HF_API_BASE = f"https://huggingface.co/api/datasets/{DATASET}/tree/main/raw"
HF_RAW_BASE = f"https://huggingface.co/datasets/{DATASET}/resolve/main/raw"

REVIEW_TREE_API = f"{HF_API_BASE}/review_categories"
META_TREE_API = f"{HF_API_BASE}/meta_categories"

REVIEW_RAW_BASE = f"{HF_RAW_BASE}/review_categories"
META_RAW_BASE = f"{HF_RAW_BASE}/meta_categories"

START_DATE = pd.to_datetime("2020-01-01", utc=True)

# Exclusive end date, so this includes the full day of 2022-12-31
END_DATE_EXCLUSIVE = pd.to_datetime("2023-01-01", utc=True)

# Use None for full scan.
# For quick testing, use something like 10_000.
MAX_REVIEW_ROWS_PER_CATEGORY = None
MAX_META_ROWS_PER_CATEGORY = None


# ----------------------------
# Robust request session
# ----------------------------

def build_session(retries=5):
    session = requests.Session()

    retry = Retry(
        total=retries,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )

    session.mount("https://", HTTPAdapter(max_retries=retry))
    return session


SESSION = build_session()


# ----------------------------
# Helpers
# ----------------------------

def stream_jsonl(url, max_rows=None):
    response = SESSION.get(url, stream=True, timeout=(30, 120))
    response.raise_for_status()

    n = 0

    for line in response.iter_lines():
        if not line:
            continue

        try:
            yield json.loads(line)
        except json.JSONDecodeError:
            continue

        n += 1

        if max_rows is not None and n >= max_rows:
            break


def parse_timestamp(ts):
    if ts is None:
        return None

    try:
        if isinstance(ts, (int, float)):
            return pd.to_datetime(ts, unit="ms", utc=True)

        return pd.to_datetime(ts, utc=True)

    except Exception:
        return None


def get_tree_files(api_url):
    response = SESSION.get(api_url, timeout=60)
    response.raise_for_status()
    return response.json()


def get_review_categories():
    files = get_tree_files(REVIEW_TREE_API)

    return sorted(
        item["path"].split("/")[-1].replace(".jsonl", "")
        for item in files
        if item.get("path", "").endswith(".jsonl")
    )


def get_meta_categories():
    files = get_tree_files(META_TREE_API)

    return sorted(
        item["path"].split("/")[-1].replace("meta_", "").replace(".jsonl", "")
        for item in files
        if item.get("path", "").endswith(".jsonl")
    )


# ============================================================
# Step 1:
# Find parent_asin values from User Reviews in the date range
# ============================================================

review_categories = get_review_categories()
meta_categories = get_meta_categories()

print(f"Found {len(review_categories)} review categories")
print(f"Found {len(meta_categories)} metadata categories")

review_parent_asins = set()
review_rows_in_range = 0
review_rows_missing_parent_asin = 0

for category in review_categories:
    url = f"{REVIEW_RAW_BASE}/{category}.jsonl"

    print(f"\nScanning User Reviews: {category}")

    rows_in_range_this_category = 0

    for row in tqdm(
        stream_jsonl(url, MAX_REVIEW_ROWS_PER_CATEGORY),
        desc=f"reviews: {category}",
    ):
        ts = parse_timestamp(row.get("timestamp"))

        if ts is None:
            continue

        if START_DATE <= ts < END_DATE_EXCLUSIVE:
            review_rows_in_range += 1
            rows_in_range_this_category += 1

            parent_asin = row.get("parent_asin")

            if parent_asin:
                review_parent_asins.add(parent_asin)
            else:
                review_rows_missing_parent_asin += 1

    print(f"Rows in date range: {rows_in_range_this_category}")

print("\nFinished User Reviews scan.")
print(f"Total review rows from 2020-01-01 to 2022-12-31: {review_rows_in_range:,}")
print(f"Unique parent_asin from those reviews: {len(review_parent_asins):,}")
print(f"Review rows missing parent_asin: {review_rows_missing_parent_asin:,}")


# ============================================================
# Step 2:
# Count Item Metadata rows matching those parent_asin values
# ============================================================

matched_metadata_rows = 0
matched_metadata_parent_asins = set()

for category in meta_categories:
    url = f"{META_RAW_BASE}/meta_{category}.jsonl"

    print(f"\nScanning Item Metadata: meta_{category}")

    matched_this_category = 0

    for row in tqdm(
        stream_jsonl(url, MAX_META_ROWS_PER_CATEGORY),
        desc=f"metadata: {category}",
    ):
        parent_asin = row.get("parent_asin")

        if parent_asin in review_parent_asins:
            matched_metadata_rows += 1
            matched_this_category += 1
            matched_metadata_parent_asins.add(parent_asin)

    print(f"Matched metadata rows: {matched_this_category}")


# ============================================================
# Results
# ============================================================

missing_metadata_parent_asins = review_parent_asins - matched_metadata_parent_asins

print("\n" + "=" * 80)
print("RESULTS")
print("=" * 80)

print(f"Review rows from 2020-01-01 to 2022-12-31: {review_rows_in_range:,}")
print(f"Unique parent_asin from those reviews: {len(review_parent_asins):,}")

print(f"\nItem Metadata rows matching those parent_asin values: {matched_metadata_rows:,}")
print(f"Unique matched metadata parent_asin: {len(matched_metadata_parent_asins):,}")

print(f"\nReview parent_asin values missing from Item Metadata: {len(missing_metadata_parent_asins):,}")

print("\nSample missing parent_asin values:")
print(sorted(list(missing_metadata_parent_asins))[:50])

/opt/homebrew/Caskroom/miniconda/base/envs/rcd_proj01/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 34 review categories
Found 34 metadata categories

Scanning User Reviews: All_Beauty


reviews: All_Beauty: 701528it [00:19, 35793.89it/s]


Rows in date range: 313231

Scanning User Reviews: Amazon_Fashion


reviews: Amazon_Fashion: 491it [02:11,  3.73it/s]


ChunkedEncodingError: ('Connection broken: IncompleteRead(287112 bytes read, 1051037619 more expected)', IncompleteRead(287112 bytes read, 1051037619 more expected))